# LLM vs Dictionary vs FAERS: Weight-Loss Medication Adverse Event Signal Detection

**Study Purpose**: Demonstrate that Gemini LLM-based AE extraction from Reddit is superior in coverage, captures under-reported signals, and detects AE signals earlier than FAERS formal pharmacovigilance.

## Hypotheses
- **H1 (Coverage)**: LLM extracts significantly more unique drug-AE pairs than dictionary-based NLP.
- **H2 (Underreporting)**: LLM captures AEs absent or under-represented in FAERS.
- **H3 (Early Detection)**: Reddit-derived LLM signals emerge temporally earlier than FAERS reports.

**Data period**: 2021–2025 | **Drug class**: Weight-Loss Medications | **Source**: Reddit (n posts) + FAERS

In [5]:
!pip install pandas matplotlib seaborn scipy statsmodels scikit-learn

  Using cached scikit_learn-1.8.0-cp314-cp314-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.8.0-cp314-cp314-macosx_12_0_arm64.whl (8.1 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]


---
## Section 0: Environment Setup and Data Loading

In [6]:
# ── 0.1 Imports ──────────────────────────────────────────────────────────────
import ast
import re
import os
from pathlib import Path
import warnings
import difflib
from collections import Counter
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu, wilcoxon, chi2_contingency, fisher_exact, kruskal
from statsmodels.stats.multitest import multipletests
from statsmodels.formula.api import ols
import statsmodels.api as sm
from sklearn.metrics import cohen_kappa_score

warnings.filterwarnings('ignore')
np.random.seed(42)

# Plot style
plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'figure.facecolor': 'white'
})

BASE = str(Path.cwd())  # project root
RESULTS = BASE
OUT = os.path.join(BASE, 'data', 'AE_Signal_Detection_Results')
os.makedirs(OUT, exist_ok=True)
print('Environment ready. Base path:', BASE)
print('Results folder:  ', OUT)

Environment ready. Base path: /Users/alexandarvincentpaulraj/Projects/TheEx/Weight Loss Meds
Results folder:   /Users/alexandarvincentpaulraj/Projects/TheEx/Weight Loss Meds/data/AE_Signal_Detection_Results


In [ ]:
# ── 0.2 Helper: safe list parser for LLM fields ───────────────────────────────
def safe_parse_list(val):
    """Parse a stringified list/comma-separated string into a Python list."""
    if pd.isna(val) or str(val).strip() in ('', 'nan', 'none', 'None', '[]'):
        return []
    s = str(val).strip()
    # Try ast.literal_eval first (for Python list literals)
    try:
        result = ast.literal_eval(s)
        if isinstance(result, list):
            return [str(x).strip().lower() for x in result if str(x).strip()]
        return [str(result).strip().lower()]
    except Exception:
        pass
    # Fallback: split by comma or semicolon
    items = re.split(r'[;,]', s)
    return [x.strip().lower() for x in items if x.strip()]

def normalise_str(s):
    """Lowercase, strip, remove excess whitespace."""
    return re.sub(r'\s+', ' ', str(s).strip().lower())

print('Helper functions defined.')

In [ ]:
# ── 0.3 Load Raw Data ─────────────────────────────────────────────────────────
reddit_raw = pd.read_csv(os.path.join(BASE, 'gemini_results_combined.csv'), low_memory=False)
drug_list_df  = pd.read_csv(os.path.join(BASE, '1.Weight-loss medications_drug_list.csv'))
ae_dict_df    = pd.read_csv(os.path.join(BASE, 'data', '2.adverse_reactions_to_organ_system.csv'))
faers_raw     = pd.read_csv(os.path.join(BASE, 'data', '4.FAERS_data.csv'))

print('=== RAW DATA SHAPES ===')
print(f'Reddit (LLM):   {reddit_raw.shape}')
print(f'Drug list:      {drug_list_df.shape}')
print(f'AE dictionary:  {ae_dict_df.shape}')
print(f'FAERS:          {faers_raw.shape}')

print('\n=== REDDIT COLUMNS ===')
print(reddit_raw.columns.tolist())
print('\n=== FAERS COLUMNS ===')
print(faers_raw.columns.tolist())

In [ ]:
# ── 0.4 Clean Reddit Data ─────────────────────────────────────────────────────
reddit = reddit_raw.copy()
n0 = len(reddit)

# Standardise column names
reddit.columns = [c.strip() for c in reddit.columns]

# Parse date
reddit['created_date'] = pd.to_datetime(reddit['created_date'], errors='coerce')
n_bad_date = reddit['created_date'].isna().sum()
reddit = reddit.dropna(subset=['created_date'])

# Filter 2021–2025
reddit = reddit[(reddit['created_date'].dt.year >= 2021) &
                (reddit['created_date'].dt.year <= 2025)]
n_date_filter = len(reddit)

# Remove deleted authors
reddit = reddit[~reddit['author'].isin(['[deleted]', '[removed]', 'AutoModerator'])]
reddit = reddit[reddit['author'].notna()]
n_author_filter = len(reddit)

# Remove duplicate post IDs (keep first)
reddit = reddit.drop_duplicates(subset=['reddit_post_id'], keep='first')
n_dedup = len(reddit)

# Reset index
reddit = reddit.reset_index(drop=True)

print('=== DATA QUALITY REPORT — REDDIT ===')
print(f'Raw records:              {n0:>8,}')
print(f'Removed (bad date):       {n_bad_date:>8,}')
print(f'After date filter:        {n_date_filter:>8,}')
print(f'After author filter:      {n_author_filter:>8,}')
print(f'After deduplication:      {n_dedup:>8,}  ← final')
print(f'Date range: {reddit["created_date"].min().date()} → {reddit["created_date"].max().date()}')
print(f'Unique authors:           {reddit["author"].nunique():>8,}')
print(f'Unique subreddits:        {reddit["subreddit"].nunique():>8,}')

reddit.to_csv(os.path.join(OUT, 'reddit_clean.csv'), index=False)
print('\nSaved: reddit_clean.csv')

In [ ]:
# ── 0.5 Clean FAERS Data ──────────────────────────────────────────────────────
faers = faers_raw.copy()
faers.columns = [c.strip() for c in faers.columns]

# Standardise column names to lowercase
faers.rename(columns={
    'Drug': 'drug_raw',
    'Year': 'year',
    'adverse Reaction': 'ae_raw',
    'Count': 'faers_count'
}, inplace=True)

faers['drug_raw'] = faers['drug_raw'].astype(str).str.strip()
faers['ae_raw'] = faers['ae_raw'].astype(str).str.strip()
faers = faers.dropna(subset=['drug_raw', 'ae_raw', 'faers_count'])
faers['faers_count'] = pd.to_numeric(faers['faers_count'], errors='coerce').fillna(0).astype(int)

print('=== DATA QUALITY REPORT — FAERS ===')
print(f'Records: {len(faers):,}')
print(f'Year range: {faers["year"].min()} – {faers["year"].max()}')
print(f'Unique drugs:  {faers["drug_raw"].nunique():,}')
print(f'Unique AEs:    {faers["ae_raw"].nunique():,}')
print(faers.head(3))

faers.to_csv(os.path.join(OUT, 'faers_clean.csv'), index=False)
print('\nSaved: faers_clean.csv')
faers = faers[faers['drug_raw'].isin(primary_drugs)]


---
## Section 1: Drug Normalisation and Reference Mapping

In [ ]:

# ── 1.1 Build Drug Normalisation Dictionary ────────────────────────────────────
drug_list_df.columns = [c.strip() for c in drug_list_df.columns]

brand2generic = {}
generic2cat = {}

glp1 = {'semaglutide', 'tirzepatide', 'liraglutide', 'dulaglutide', 'exenatide', 'lixisenatide', 'albiglutide'}
sglt2 = {'canagliflozin', 'dapagliflozin', 'empagliflozin', 'ertugliflozin'}
stim = {'phentermine', 'benzphetamine', 'diethylpropion', 'phendimetrazine'}
primary_drugs = glp1 | sglt2 | stim | {'orlistat', 'naltrexone', 'bupropion', 'topiramate', 'lorcaserin', 'metformin'}

def wl_category(generic):
    if generic in glp1: return 'GLP-1 RA'
    if generic in sglt2: return 'SGLT2 inhibitor'
    if generic in stim: return 'Stimulant'
    return 'Other weight-loss'

for _, row in drug_list_df.iterrows():
    generic = normalise_str(row.get('generic_name', ''))
    variant = normalise_str(row.get('variant', ''))
    if not generic:
        continue
    brand2generic[generic] = generic
    generic2cat[generic] = wl_category(generic)
    if variant:
        brand2generic[variant] = generic

all_generics = set(generic2cat.keys())
print(f'Reference drugs (generics):  {len(all_generics)}')
print(f'Drug name aliases mapped:    {len(brand2generic)}')
print(f'Drug categories:             {sorted(set(generic2cat.values()))}')


In [ ]:
# ── 1.2 Build AE Dictionary Lookup ────────────────────────────────────────────
ae_dict_df.columns = [c.strip() for c in ae_dict_df.columns]

ae2organ = {}   # ae_term → organ_system
for _, row in ae_dict_df.iterrows():
    term = normalise_str(row.get('Adverse reactions', ''))
    organ = str(row.get('Organ System', 'Unknown')).strip()
    if term:
        ae2organ[term] = organ

ae_dict_terms = set(ae2organ.keys())
print(f'AE dictionary terms: {len(ae_dict_terms)}')
print(f'Organ systems: {sorted(set(ae2organ.values()))}')

In [ ]:
# ── 1.3 Normalise FAERS Drug and AE Fields ────────────────────────────────────
faers['drug'] = faers['drug_raw'].str.lower().str.strip().map(
    lambda x: brand2generic.get(x, x)   # map brand→generic where possible
)
faers['ae']   = faers['ae_raw'].str.lower().str.strip()
faers['category'] = faers['drug'].map(generic2cat).fillna('Unknown')

# Aggregate by (drug, ae, year)
faers_norm = (faers.groupby(['drug', 'ae', 'year'], as_index=False)
              ['faers_count'].sum())

# First FAERS year per (drug, ae) pair
faers_first = (faers_norm.groupby(['drug', 'ae'])['year']
               .min().reset_index(name='first_year_faers'))

# Total count per (drug, ae) across all years
faers_total = (faers_norm.groupby(['drug', 'ae'], as_index=False)
               ['faers_count'].sum())
faers_total = faers_total.merge(faers_first, on=['drug', 'ae'])

faers_norm.to_csv(os.path.join(OUT, 'faers_normalised.csv'), index=False)
print(f'FAERS normalised records:   {len(faers_norm):,}')
print(f'FAERS unique drug-AE pairs: {len(faers_total):,}')
print(f'FAERS year range:           {faers_norm["year"].min()} – {faers_norm["year"].max()}')
print('\nSaved: faers_normalised.csv')

---
## Section 2: LLM-Based AE Extraction (Method A)

In [ ]:
# ── 2.1 Parse LLM Fields ──────────────────────────────────────────────────────
# The Gemini LLM has already extracted adverse_events and relevant_drugs per post

reddit['drugs_llm_raw'] = reddit['relevant_drugs'].apply(safe_parse_list)
reddit['aes_llm_raw']   = reddit['adverse_events'].apply(safe_parse_list)

n_has_drug = (reddit['drugs_llm_raw'].apply(len) > 0).sum()
n_has_ae   = (reddit['aes_llm_raw'].apply(len)   > 0).sum()
n_has_both = ((reddit['drugs_llm_raw'].apply(len) > 0) &
              (reddit['aes_llm_raw'].apply(len)   > 0)).sum()

print(f'Total clean posts:           {len(reddit):,}')
print(f'Posts with ≥1 LLM drug:      {n_has_drug:,} ({100*n_has_drug/len(reddit):.1f}%)')
print(f'Posts with ≥1 LLM AE:        {n_has_ae:,} ({100*n_has_ae/len(reddit):.1f}%)')
print(f'Posts with ≥1 drug AND ≥1 AE: {n_has_both:,} ({100*n_has_both/len(reddit):.1f}%)')

In [ ]:
# ── 2.2 Normalise LLM Drug Names ──────────────────────────────────────────────
def normalise_drug_list(drug_list):
    """Map each drug name to canonical generic; keep only known weight-losss."""
    result = []
    for d in drug_list:
        d = normalise_str(d)
        # Direct lookup
        if d in brand2generic:
            result.append(brand2generic[d])
        else:
            # Partial match: check if any known name is contained in the term
            matched = [g for g in all_generics if g in d or d in g]
            if matched:
                result.append(matched[0])
    return list(set(result))  # deduplicate

reddit['drugs_llm'] = reddit['drugs_llm_raw'].apply(normalise_drug_list)

n_known_drug = (reddit['drugs_llm'].apply(len) > 0).sum()
print(f'Posts with ≥1 recognised weight-loss (LLM): {n_known_drug:,} ({100*n_known_drug/len(reddit):.1f}%)')

In [ ]:
# ── 2.3 Normalise LLM AE Terms and Map to Organ System ───────────────────────
def normalise_ae_llm(ae_list):
    """Normalise LLM AE terms; map to organ system (or 'Novel/Unlabelled')."""
    result = []
    for ae in ae_list:
        ae = normalise_str(ae)
        ae = re.sub(r'[^a-z0-9\s\-]', ' ', ae).strip()
        if not ae:
            continue
        # Exact dict match
        if ae in ae_dict_terms:
            organ = ae2organ[ae]
            result.append({'ae': ae, 'organ_system': organ, 'dict_matched': True})
        else:
            # Fuzzy match
            close = difflib.get_close_matches(ae, ae_dict_terms, n=1, cutoff=0.85)
            if close:
                result.append({'ae': ae, 'organ_system': ae2organ[close[0]], 'dict_matched': True})
            else:
                result.append({'ae': ae, 'organ_system': 'Novel/Unlabelled', 'dict_matched': False})
    return result

reddit['aes_llm_parsed'] = reddit['aes_llm_raw'].apply(normalise_ae_llm)
print('LLM AE normalisation complete.')
print('Sample AE parse:', reddit[reddit['aes_llm_parsed'].apply(len) > 0]['aes_llm_parsed'].iloc[0])

In [ ]:
# ── 2.4 Build LLM Drug-AE Longitudinal Table ──────────────────────────────────
llm_records = []

for _, row in reddit.iterrows():
    drugs = row['drugs_llm']
    aes   = row['aes_llm_parsed']
    if not drugs or not aes:
        continue
    for drug in drugs:
        for ae_info in aes:
            llm_records.append({
                'reddit_post_id':       row['reddit_post_id'],
                'created_date':         row['created_date'],
                'author':               row['author'],
                'subreddit':            row['subreddit'],
                'drug':                 drug,
                'category':             generic2cat.get(drug, 'Unknown'),
                'ae_llm':               ae_info['ae'],
                'organ_system_llm':     ae_info['organ_system'],
                'dict_matched':         ae_info['dict_matched'],
                'indication':           str(row.get('indication', '')),
                'outcome':              str(row.get('outcome', '')),
                'causality_hints':      str(row.get('causality_hints', '')),
                'sentiment_perceptions':str(row.get('sentiment_perceptions', '')),
                'off_label_use':        str(row.get('off_label_use', '')),
                'drug_drug_interaction':str(row.get('drug_drug_interaction', '')),
            })

llm_long = pd.DataFrame(llm_records)
llm_long = llm_long.drop_duplicates(subset=['reddit_post_id', 'drug', 'ae_llm'])
llm_long = llm_long.reset_index(drop=True)
llm_long.to_csv(os.path.join(OUT, 'llm_drug_ae_longitudinal.csv'), index=False)

print('=== LLM EXTRACTION SUMMARY ===')
print(f'Total LLM drug-AE-post triplets: {len(llm_long):,}')
print(f'Unique drug-AE pairs (LLM):      {llm_long.groupby(["drug","ae_llm"]).ngroups:,}')
print(f'Unique drugs (LLM):              {llm_long["drug"].nunique()}')
print(f'Unique AE terms (LLM):           {llm_long["ae_llm"].nunique():,}')
print(f'AEs matched to dictionary:       {llm_long["dict_matched"].sum():,} ({100*llm_long["dict_matched"].mean():.1f}%)')
print(f'Novel AEs (not in dictionary):   {(~llm_long["dict_matched"]).sum():,}')
llm_long.head(3)

---
## Section 3: Dictionary-Based AE Extraction (Method B)

In [ ]:
# ── 3.1 Preprocess Text and Extract AEs by String Matching ────────────────────
# Sort terms longest-first to match multi-word terms before sub-terms
sorted_ae_terms  = sorted(ae_dict_terms, key=len, reverse=True)
sorted_drug_keys = sorted(brand2generic.keys(), key=len, reverse=True)

def extract_dict_aes(text):
    text = normalise_str(text)
    text = re.sub(r'[^a-z0-9\s\-]', ' ', text)
    found = [term for term in sorted_ae_terms if re.search(r'\b' + re.escape(term) + r'\b', text)]
    return list(set(found))

def extract_dict_drugs(text):
    text = normalise_str(text)
    text = re.sub(r'[^a-z0-9\s\-]', ' ', text)
    found = []
    for key in sorted_drug_keys:
        if re.search(r'\b' + re.escape(key) + r'\b', text):
            generic = brand2generic[key]
            if generic not in found:
                found.append(generic)
    return found

print('Applying dictionary extraction (may take a few minutes on large datasets)...')
reddit['aes_dict']   = reddit['combined_text'].apply(extract_dict_aes)
reddit['drugs_dict'] = reddit['combined_text'].apply(extract_dict_drugs)
print('Dictionary extraction complete.')

n_dict_both = ((reddit['drugs_dict'].apply(len) > 0) &
               (reddit['aes_dict'].apply(len)   > 0)).sum()
print(f'Posts with ≥1 drug AND ≥1 AE (dictionary): {n_dict_both:,} ({100*n_dict_both/len(reddit):.1f}%)')

In [ ]:
# ── 3.2 Build Dictionary Drug-AE Longitudinal Table ───────────────────────────
dict_records = []
for _, row in reddit.iterrows():
    drugs = row['drugs_dict']
    aes   = row['aes_dict']
    if not drugs or not aes:
        continue
    for drug in drugs:
        for ae in aes:
            dict_records.append({
                'reddit_post_id': row['reddit_post_id'],
                'created_date':   row['created_date'],
                'author':         row['author'],
                'subreddit':      row['subreddit'],
                'drug':           drug,
                'category':       generic2cat.get(drug, 'Unknown'),
                'ae_dict':        ae,
                'organ_system_dict': ae2organ.get(ae, 'Unknown'),
            })

dict_long = pd.DataFrame(dict_records)
dict_long = dict_long.drop_duplicates(subset=['reddit_post_id', 'drug', 'ae_dict'])
dict_long = dict_long.reset_index(drop=True)
dict_long.to_csv(os.path.join(OUT, 'dict_drug_ae_longitudinal.csv'), index=False)

print('=== DICTIONARY EXTRACTION SUMMARY ===')
print(f'Total dict drug-AE-post triplets: {len(dict_long):,}')
print(f'Unique drug-AE pairs (dict):      {dict_long.groupby(["drug","ae_dict"]).ngroups:,}')
print(f'Unique drugs (dict):              {dict_long["drug"].nunique()}')
print(f'Unique AE terms (dict):           {dict_long["ae_dict"].nunique():,}')
dict_long.head(3)

---
## Section 4: H1 — LLM vs Dictionary Coverage Comparison

In [ ]:
# ── 4.1 Unique AE Terms and Drug-AE Pair Comparison ───────────────────────────
set_ae_llm  = set(llm_long['ae_llm'].str.lower().unique())
set_ae_dict = set(dict_long['ae_dict'].str.lower().unique())

set_pair_llm  = set(zip(llm_long['drug'], llm_long['ae_llm']))
set_pair_dict = set(zip(dict_long['drug'], dict_long['ae_dict']))

print('=== UNIQUE AE TERM COMPARISON ===')
print(f'LLM unique AE terms:              {len(set_ae_llm):>6,}')
print(f'Dictionary unique AE terms:       {len(set_ae_dict):>6,}')
print(f'Shared:                           {len(set_ae_llm & set_ae_dict):>6,}')
print(f'LLM-only (novel):                 {len(set_ae_llm - set_ae_dict):>6,}')
print(f'Dictionary-only:                  {len(set_ae_dict - set_ae_llm):>6,}')

print('\n=== UNIQUE DRUG-AE PAIR COMPARISON ===')
print(f'LLM unique drug-AE pairs:         {len(set_pair_llm):>6,}')
print(f'Dictionary unique drug-AE pairs:  {len(set_pair_dict):>6,}')
print(f'Shared:                           {len(set_pair_llm & set_pair_dict):>6,}')
print(f'LLM-only:                         {len(set_pair_llm - set_pair_dict):>6,}')
print(f'Dictionary-only:                  {len(set_pair_dict - set_pair_llm):>6,}')

# Save comparison table
coverage_df = pd.DataFrame([
    {'Metric': 'Unique AE Terms',      'LLM': len(set_ae_llm),   'Dictionary': len(set_ae_dict),
     'Shared': len(set_ae_llm & set_ae_dict), 'LLM_only': len(set_ae_llm - set_ae_dict)},
    {'Metric': 'Unique Drug-AE Pairs', 'LLM': len(set_pair_llm), 'Dictionary': len(set_pair_dict),
     'Shared': len(set_pair_llm & set_pair_dict), 'LLM_only': len(set_pair_llm - set_pair_dict)},
])
coverage_df.to_csv(os.path.join(OUT, 'unique_pairs_comparison.csv'), index=False)
print('\nSaved: unique_pairs_comparison.csv')

In [ ]:
# ── 4.2 Statistical Test — Post-Level AE Counts ───────────────────────────────
# For posts processed by both methods, compare AE counts (paired test)

post_llm  = llm_long.groupby('reddit_post_id')['ae_llm'].count().rename('ae_count_llm')
post_dict = dict_long.groupby('reddit_post_id')['ae_dict'].count().rename('ae_count_dict')
post_both = pd.concat([post_llm, post_dict], axis=1).dropna()

if len(post_both) >= 10:
    stat, p = wilcoxon(post_both['ae_count_llm'], post_both['ae_count_dict'])
    n = len(post_both)
    r = stat / (n * (n + 1) / 4)   # rank-biserial r approximation
    print(f'Posts with both methods: {n:,}')
    print(f'Median AE/post (LLM):        {post_both["ae_count_llm"].median():.1f}')
    print(f'Median AE/post (Dictionary): {post_both["ae_count_dict"].median():.1f}')
    print(f'Paired Wilcoxon: W={stat:.1f}, p={p:.4g}')
    print(f'Effect size (rank-biserial r): {r:.3f}')
else:
    print('Insufficient overlapping posts for paired test. Reporting marginal counts.')
    print(f'Median AE/post (LLM):        {llm_long.groupby("reddit_post_id")["ae_llm"].count().median():.1f}')
    print(f'Median AE/post (Dictionary): {dict_long.groupby("reddit_post_id")["ae_dict"].count().median():.1f}')

In [ ]:
# ── 4.3 Figure 1: Venn Diagram — Drug-AE Pairs (LLM vs Dictionary) ───────────
# Proportional rectangle Venn (no external dependency required)

L = len(set_pair_llm)
D = len(set_pair_dict)
I = len(set_pair_llm & set_pair_dict)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: AE Terms
ax = axes[0]
la, da, ia = len(set_ae_llm), len(set_ae_dict), len(set_ae_llm & set_ae_dict)
circle1 = plt.Circle((0.37, 0.5), 0.35, color='steelblue', alpha=0.45, label=f'LLM ({la:,})')
circle2 = plt.Circle((0.63, 0.5), 0.35, color='darkorange', alpha=0.45, label=f'Dictionary ({da:,})')
ax.add_patch(circle1); ax.add_patch(circle2)
ax.text(0.25, 0.5, str(la - ia), ha='center', va='center', fontsize=13, fontweight='bold')
ax.text(0.75, 0.5, str(da - ia), ha='center', va='center', fontsize=13, fontweight='bold')
ax.text(0.5,  0.5, str(ia),      ha='center', va='center', fontsize=13, fontweight='bold')
ax.set_xlim(0, 1); ax.set_ylim(0.05, 0.95); ax.axis('off')
ax.set_title('Unique AE Terms', fontweight='bold')
ax.legend(handles=[
    mpatches.Patch(color='steelblue', alpha=0.6, label=f'LLM ({la:,})'),
    mpatches.Patch(color='darkorange', alpha=0.6, label=f'Dictionary ({da:,})')
], loc='lower center')

# Right: Drug-AE Pairs
ax = axes[1]
circle1 = plt.Circle((0.37, 0.5), 0.35, color='steelblue', alpha=0.45)
circle2 = plt.Circle((0.63, 0.5), 0.35, color='darkorange', alpha=0.45)
ax.add_patch(circle1); ax.add_patch(circle2)
ax.text(0.25, 0.5, str(L - I), ha='center', va='center', fontsize=13, fontweight='bold')
ax.text(0.75, 0.5, str(D - I), ha='center', va='center', fontsize=13, fontweight='bold')
ax.text(0.5,  0.5, str(I),     ha='center', va='center', fontsize=13, fontweight='bold')
ax.set_xlim(0, 1); ax.set_ylim(0.05, 0.95); ax.axis('off')
ax.set_title('Unique Drug-AE Pairs', fontweight='bold')
ax.legend(handles=[
    mpatches.Patch(color='steelblue', alpha=0.6, label=f'LLM ({L:,})'),
    mpatches.Patch(color='darkorange', alpha=0.6, label=f'Dictionary ({D:,})')
], loc='lower center')

fig.suptitle('Figure 1: LLM vs Dictionary — AE Coverage Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig1_venn_llm_vs_dict.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ── 4.4 Figure: Top 30 AEs by Method (Grouped Bar) ────────────────────────────
top_ae_llm  = llm_long['ae_llm'].value_counts().head(30)
top_ae_dict = dict_long['ae_dict'].value_counts().head(30)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, counts, title, color in [
    (axes[0], top_ae_llm,  'LLM-Based (Top 30)',        'steelblue'),
    (axes[1], top_ae_dict, 'Dictionary-Based (Top 30)', 'darkorange'),
]:
    ax.barh(counts.index[::-1], np.log1p(counts.values[::-1]), color=color, alpha=0.8)
    ax.set_xlabel('log(Count + 1)')
    ax.set_title(title, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
fig.suptitle('Top 30 Adverse Events by Extraction Method', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig_top_aes_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ── 4.5 Novel LLM-Only AEs (not in dictionary and not in FAERS) ───────────────
faers_ae_set = set(faers_norm['ae'].str.lower().unique())

# AEs from LLM that are not in the dictionary AND not in FAERS
novel_llm_aes = set_ae_llm - set_ae_dict - faers_ae_set
novel_counts  = llm_long[llm_long['ae_llm'].isin(novel_llm_aes)]['ae_llm'].value_counts().reset_index()
novel_counts.columns = ['ae', 'post_count']
novel_counts['unique_authors'] = (
    llm_long[llm_long['ae_llm'].isin(novel_llm_aes)]
    .groupby('ae_llm')['author'].nunique().reindex(novel_counts['ae']).values
)
novel_counts.to_csv(os.path.join(OUT, 'novel_llm_only_aes.csv'), index=False)

print(f'Novel AEs (LLM-only, absent from dictionary AND FAERS): {len(novel_llm_aes):,}')
print('Top 20 novel AEs by post count:')
print(novel_counts.head(20).to_string(index=False))

---
## Section 5: H2 — FAERS Comparison, PRR, and Underreported AEs

In [ ]:
# ── 5.1 Three-Way Venn: LLM vs Dictionary vs FAERS ───────────────────────────
set_pair_faers = set(zip(faers_total['drug'].str.lower(), faers_total['ae'].str.lower()))

# Normalise LLM and dict pairs to lowercase for fair comparison
sp_llm  = set((d.lower(), a.lower()) for d, a in set_pair_llm)
sp_dict = set((d.lower(), a.lower()) for d, a in set_pair_dict)
sp_f    = set_pair_faers

venn_counts = {
    'LLM ∩ Dict ∩ FAERS':  len(sp_llm & sp_dict & sp_f),
    'LLM ∩ Dict only':      len(sp_llm & sp_dict - sp_f),
    'LLM ∩ FAERS only':     len(sp_llm & sp_f - sp_dict),
    'Dict ∩ FAERS only':    len(sp_dict & sp_f - sp_llm),
    'LLM only':             len(sp_llm - sp_dict - sp_f),
    'Dict only':            len(sp_dict - sp_llm - sp_f),
    'FAERS only':           len(sp_f - sp_llm - sp_dict),
    'Total LLM':            len(sp_llm),
    'Total Dict':           len(sp_dict),
    'Total FAERS':          len(sp_f),
}
venn_df = pd.DataFrame(list(venn_counts.items()), columns=['Region', 'Count'])
venn_df.to_csv(os.path.join(OUT, 'three_way_venn_counts.csv'), index=False)

print('=== THREE-WAY PAIR OVERLAP ===')
print(venn_df.to_string(index=False))

In [ ]:
# ── 5.2 Underreported AE Identification ───────────────────────────────────────
# Aggregate LLM data: count posts and unique authors per (drug, ae)
MIN_POSTS   = 10
MIN_AUTHORS = 3

llm_agg = (
    llm_long.groupby(['drug', 'ae_llm'])
    .agg(reddit_post_count=('reddit_post_id', 'nunique'),
         unique_authors=('author', 'nunique'))
    .reset_index()
    .rename(columns={'ae_llm': 'ae'})
)

# Merge with FAERS totals
underrep = llm_agg.merge(
    faers_total[['drug', 'ae', 'faers_count', 'first_year_faers']],
    on=['drug', 'ae'], how='left'
)
underrep['faers_count']      = underrep['faers_count'].fillna(0).astype(int)
underrep['first_year_faers'] = underrep['first_year_faers'].fillna('Absent')

# Filter to signal-strength threshold
threshold_mask = (
    (underrep['reddit_post_count'] >= MIN_POSTS) &
    (underrep['unique_authors']    >= MIN_AUTHORS)
)
underrep_filtered = underrep[threshold_mask].copy()

# Classify
underrep_filtered['faers_status'] = underrep_filtered.apply(
    lambda r: 'FAERS-absent' if r['faers_count'] == 0
              else ('Highly-underreported' if r['reddit_post_count'] / (r['faers_count'] + 1) > 10
                    else 'Reported'),
    axis=1
)
underrep_filtered = underrep_filtered.sort_values('reddit_post_count', ascending=False)
underrep_filtered.to_csv(os.path.join(OUT, 'underreported_ae_llm_vs_faers.csv'), index=False)

print(f'=== UNDERREPORTED AE ANALYSIS (threshold: ≥{MIN_POSTS} posts, ≥{MIN_AUTHORS} authors) ===')
print(underrep_filtered['faers_status'].value_counts().to_string())
print(f'\nTotal LLM drug-AE pairs above threshold: {len(underrep_filtered):,}')
print('\nTop 20 underreported (FAERS-absent or highly-underreported):')
cols_show = ['drug', 'ae', 'reddit_post_count', 'unique_authors', 'faers_count', 'first_year_faers', 'faers_status']
print(underrep_filtered[underrep_filtered['faers_status'] != 'Reported'][cols_show].head(20).to_string(index=False))

In [ ]:
# ── 5.3 Figure: Top 25 Underreported AEs — Diverging Bar Chart ───────────────
top_under = (
    underrep_filtered[underrep_filtered['faers_status'] != 'Reported']
    .sort_values('reddit_post_count', ascending=False)
    .head(25)
)

fig, ax = plt.subplots(figsize=(12, 9))
y     = np.arange(len(top_under))
label = top_under['drug'] + ' | ' + top_under['ae']

ax.barh(y, -np.log1p(top_under['reddit_post_count']),
        color='steelblue', alpha=0.85, label='Reddit / LLM (log)')
ax.barh(y,  np.log1p(top_under['faers_count']),
        color='darkorange', alpha=0.85, label='FAERS (log)')
ax.set_yticks(y); ax.set_yticklabels(label, fontsize=9)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('← log(Reddit count+1)  |  log(FAERS count+1) →')
ax.set_title('Figure 3: Top 25 Underreported Drug-AE Pairs\n(LLM-detected, absent/sparse in FAERS)',
             fontweight='bold')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig3_underreported_diverging.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5.4 Proportional Reporting Ratio (PRR) Signal Detection ──────────────────
# PRR = (ae_count_drug / total_ae_drug) / (ae_count_others / total_ae_others)
# Signal criteria: PRR ≥ 2, chi² p < 0.05, N ≥ 3

# Use post counts as the frequency metric
llm_counts_pivot = (
    llm_long.groupby(['drug', 'ae_llm'])['reddit_post_id']
    .nunique().reset_index(name='n_ae_drug')
    .rename(columns={'ae_llm': 'ae'})
)

prr_results = []
total_all = llm_counts_pivot['n_ae_drug'].sum()

for _, row in llm_counts_pivot.iterrows():
    drug, ae, n_ae_drug = row['drug'], row['ae'], row['n_ae_drug']
    if n_ae_drug < 3:
        continue
    n_drug_total = llm_counts_pivot[llm_counts_pivot['drug'] == drug]['n_ae_drug'].sum()
    n_ae_other   = llm_counts_pivot[(llm_counts_pivot['ae'] == ae) &
                                    (llm_counts_pivot['drug'] != drug)]['n_ae_drug'].sum()
    n_other_total = total_all - n_drug_total

    if n_drug_total == 0 or n_other_total == 0:
        continue

    # 2×2 contingency
    a = n_ae_drug
    b = n_drug_total - n_ae_drug
    c = n_ae_other
    d = n_other_total - n_ae_other

    if min(a, b, c, d) < 0:
        continue

    prr = (a / (a + b + 1e-9)) / (c / (c + d + 1e-9))
    table = np.array([[a, b], [c, d]])
    try:
        chi2, p_val, _, _ = chi2_contingency(table, correction=True)
    except Exception:
        continue

    prr_results.append({
        'drug': drug, 'ae': ae, 'n_ae_drug': n_ae_drug,
        'prr': round(prr, 3), 'chi2': round(chi2, 3), 'p_value': p_val,
        'signal': (prr >= 2.0) and (p_val < 0.05) and (n_ae_drug >= 3)
    })

prr_df = pd.DataFrame(prr_results)

# FDR correction (Benjamini-Hochberg)
if len(prr_df) > 0:
    _, q_vals, _, _ = multipletests(prr_df['p_value'], method='fdr_bh')
    prr_df['q_value'] = q_vals
    prr_df['signal_fdr'] = (prr_df['prr'] >= 2.0) & (prr_df['q_value'] < 0.05) & (prr_df['n_ae_drug'] >= 3)

prr_df.to_csv(os.path.join(OUT, 'prr_llm_signals.csv'), index=False)
n_signals = prr_df['signal_fdr'].sum() if 'signal_fdr' in prr_df.columns else 0

print(f'Total PRR tests run:      {len(prr_df):,}')
print(f'Signals (PRR≥2, q<0.05): {n_signals:,}')
print('\nTop 20 PRR signals:')
print(prr_df[prr_df.get('signal_fdr', prr_df.get('signal', False))]
      .sort_values('prr', ascending=False).head(20)
      [['drug', 'ae', 'n_ae_drug', 'prr', 'chi2', 'p_value', 'q_value']]
      .to_string(index=False))

---
## Section 6: H3 — Temporal Early-Signal Detection Analysis

In [ ]:
# ── 6.1 Compute Lead Time: First Reddit Date vs First FAERS Year ───────────────
llm_long['created_date'] = pd.to_datetime(llm_long['created_date'], errors='coerce')

# First Reddit date per (drug, ae) pair
first_reddit = (
    llm_long.groupby(['drug', 'ae_llm'])['created_date']
    .min().reset_index()
    .rename(columns={'ae_llm': 'ae', 'created_date': 'first_reddit_date'})
)

# Merge with first FAERS year
lead_df = first_reddit.merge(faers_first, on=['drug', 'ae'], how='inner')
lead_df['first_reddit_year'] = lead_df['first_reddit_date'].dt.year
lead_df['lead_time_years']   = lead_df['first_year_faers'] - lead_df['first_reddit_year']

lead_df.to_csv(os.path.join(OUT, 'signal_lead_time.csv'), index=False)

leads_reddit_first = (lead_df['lead_time_years'] > 0).sum()
leads_same         = (lead_df['lead_time_years'] == 0).sum()
leads_faers_first  = (lead_df['lead_time_years'] < 0).sum()

print('=== SIGNAL LEAD TIME ANALYSIS ===')
print(f'Matched drug-AE pairs (Reddit LLM ∩ FAERS): {len(lead_df):,}')
print(f'Reddit leads FAERS (lead > 0):   {leads_reddit_first:,} ({100*leads_reddit_first/len(lead_df):.1f}%)')
print(f'Same year:                       {leads_same:,} ({100*leads_same/len(lead_df):.1f}%)')
print(f'FAERS leads Reddit (lead < 0):   {leads_faers_first:,} ({100*leads_faers_first/len(lead_df):.1f}%)')
print(f'Median lead time: {lead_df["lead_time_years"].median():.1f} years')
print(f'Mean lead time:   {lead_df["lead_time_years"].mean():.2f} years')
print(f'IQR:              [{lead_df["lead_time_years"].quantile(0.25):.1f}, {lead_df["lead_time_years"].quantile(0.75):.1f}]')

In [ ]:
# ── 6.2 Statistical Test — Lead Time ──────────────────────────────────────────
# One-sample Wilcoxon signed-rank test: H0 = median lead time = 0
if len(lead_df) >= 10:
    lt = lead_df['lead_time_years'].dropna().values
    stat_w, p_w = wilcoxon(lt, zero_method='wilcox')
    print(f'Wilcoxon signed-rank test (H0: median lead time = 0):')
    print(f'  W = {stat_w:.1f}, p = {p_w:.4g}')

    # Bootstrap 95% CI for median lead time
    bootstrap_medians = [np.median(np.random.choice(lt, len(lt), replace=True))
                         for _ in range(10_000)]
    ci_lo = np.percentile(bootstrap_medians, 2.5)
    ci_hi = np.percentile(bootstrap_medians, 97.5)
    print(f'\nBootstrap 95% CI for median lead time: [{ci_lo:.2f}, {ci_hi:.2f}] years')

In [ ]:
# ── 6.3 Figure 2: Lead Time Histogram ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
ax.hist(lead_df['lead_time_years'].dropna(), bins=15, color='steelblue', edgecolor='white',
        alpha=0.85)
ax.axvline(0,  color='black',  lw=1.5, linestyle='--', label='No lead')
ax.axvline(lead_df['lead_time_years'].median(), color='red', lw=2,
           label=f'Median = {lead_df["lead_time_years"].median():.1f} yr')
ax.set_xlabel('Lead Time (years): Reddit ahead → FAERS ahead')
ax.set_ylabel('Number of drug-AE pairs')
ax.set_title('Figure 2a: Distribution of Reddit-vs-FAERS Lead Times')
ax.legend()
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# Cumulative LLM vs FAERS detection by year
ax = axes[1]
years = range(2021, 2026)
cum_llm   = [len(set(zip(llm_long[llm_long['created_date'].dt.year <= y]['drug'],
                         llm_long[llm_long['created_date'].dt.year <= y]['ae_llm'])))
             for y in years]
cum_faers = [len(set(zip(faers_norm[faers_norm['year'] <= y]['drug'],
                         faers_norm[faers_norm['year'] <= y]['ae'])))
             for y in years]

ax.plot(list(years), cum_llm,   'o-', color='steelblue',  lw=2, label='LLM (Reddit)')
ax.plot(list(years), cum_faers, 's-', color='darkorange', lw=2, label='FAERS')
ax.axvline(2020, color='grey', linestyle=':', lw=1.5, label='COVID-19 (2020)')
ax.fill_between(list(years), cum_faers, cum_llm, alpha=0.15, color='steelblue',
                label='Coverage gap')
ax.set_xlabel('Year')
ax.set_ylabel('Cumulative unique drug-AE pairs detected')
ax.set_title('Figure 2b: Cumulative Signal Detection (LLM vs FAERS)', fontweight='bold')
ax.legend()
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig2_lead_time_and_cumulative.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ── 6.4 Interrupted Time Series (ITS) Analysis — COVID-19 Intervention ────────
# Monthly Reddit post counts per drug category, segmented at March 2020

reddit['year_month'] = reddit['created_date'].dt.to_period('M')
reddit['month_num']  = ((reddit['created_date'].dt.year - 2021) * 12 +
                        reddit['created_date'].dt.month)

# Extract categories from LLM field for each post
post_cats = llm_long[['reddit_post_id', 'category']].drop_duplicates()
reddit_cat = reddit.merge(post_cats, on='reddit_post_id', how='left')
reddit_cat = reddit_cat.dropna(subset=['category'])
reddit_cat = reddit_cat[reddit_cat['category'] != 'Unknown']

its_results = []
COVID_MONTH = (2020 - 2021) * 12 + 3  # March 2020 = month 63

for cat in sorted(reddit_cat['category'].unique()):
    ts = (
        reddit_cat[reddit_cat['category'] == cat]
        .groupby('month_num')['reddit_post_id'].nunique()
        .reset_index(name='post_count')
    )
    ts['intervention']      = (ts['month_num'] >= COVID_MONTH).astype(int)
    ts['time_after_covid']  = (ts['month_num'] - COVID_MONTH).clip(lower=0)

    if len(ts) < 20:
        continue
    try:
        model  = sm.OLS.from_formula(
            'post_count ~ month_num + intervention + time_after_covid', data=ts
        ).fit(cov_type='HC3')
        c = model.params
        p = model.pvalues
        ci = model.conf_int()
        its_results.append({
            'category': cat,
            'beta_time':        round(c['month_num'], 3),
            'p_time':           round(p['month_num'], 4),
            'beta_covid_step':  round(c['intervention'], 3),
            'p_covid_step':     round(p['intervention'], 4),
            'beta_covid_slope': round(c['time_after_covid'], 3),
            'p_covid_slope':    round(p['time_after_covid'], 4),
            'r_squared':        round(model.rsquared, 3),
        })
    except Exception as e:
        print(f'ITS failed for {cat}: {e}')

its_df = pd.DataFrame(its_results)
its_df.to_csv(os.path.join(OUT, 'its_results.csv'), index=False)
print('=== INTERRUPTED TIME SERIES — COVID-19 INTERVENTION ===')
print(its_df.to_string(index=False))

In [ ]:
# ── 6.5 Kaplan-Meier Time-to-First-Detection (LLM vs FAERS) ──────────────────
try:
    from lifelines import KaplanMeierFitter
    from lifelines.statistics import logrank_test

    # Event = drug-AE pair first detected
    # Time  = year of first detection (relative to 2021)
    km_llm = (
        llm_long.groupby(['drug', 'ae_llm'])['created_date'].min()
        .reset_index(name='first_date')
    )
    km_llm['time'] = km_llm['first_date'].dt.year - 2021
    km_llm['event'] = 1  # observed

    km_faers = faers_first.copy()
    km_faers['time'] = km_faers['first_year_faers'] - 2021
    km_faers['event'] = 1

    # Clip to 0–10 (2021–2025)
    km_llm  = km_llm[(km_llm['time'] >= 0)  & (km_llm['time']  <= 10)]
    km_faers = km_faers[(km_faers['time'] >= 0) & (km_faers['time'] <= 10)]

    kmf_llm   = KaplanMeierFitter(label='Reddit / LLM')
    kmf_faers = KaplanMeierFitter(label='FAERS')

    fig, ax = plt.subplots(figsize=(10, 6))
    kmf_llm.fit(km_llm['time'],   km_llm['event'],   label='Reddit / LLM')
    kmf_faers.fit(km_faers['time'], km_faers['event'], label='FAERS')

    kmf_llm.plot_survival_function(ax=ax, ci_show=True, color='steelblue', lw=2)
    kmf_faers.plot_survival_function(ax=ax, ci_show=True, color='darkorange', lw=2)

    results_lr = logrank_test(km_llm['time'], km_faers['time'],
                              km_llm['event'], km_faers['event'])

    ax.set_xlabel('Years since 2021')
    ax.set_ylabel('Proportion of pairs NOT yet detected')
    ax.set_title(f'Figure 4: KM Time-to-First Detection\nLog-rank p = {results_lr.p_value:.4g}',
                 fontweight='bold')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, 'fig4_kaplan_meier.png'), dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Log-rank test: p = {results_lr.p_value:.4g}')
    print(f'LLM median time-to-detection:   {kmf_llm.median_survival_time_:.1f} years')
    print(f'FAERS median time-to-detection: {kmf_faers.median_survival_time_:.1f} years')

except ImportError:
    print('lifelines not installed. Run: pip install lifelines')
    print('Skipping Kaplan-Meier analysis.')

---
## Section 7: LLM Contextual Richness Analysis

In [ ]:
# ── 7.1 Outcome and Causality Profiling ───────────────────────────────────────

import textwrap

# ── Normalise outcome into clean short categories ─────────────────────────────
OUTCOME_MAP = {
    'hospitali'     : 'Hospitalised',
    'fatal'         : 'Fatal',
    'died'          : 'Fatal',
    'death'         : 'Fatal',
    'recovered'     : 'Recovered',
    'resolv'        : 'Recovered',
    'improved'      : 'Improved',
    'tapering'      : 'Tapering / Switching',
    'taper'         : 'Tapering / Switching',
    'switched'      : 'Tapering / Switching',
    'discontinu'    : 'Discontinued',
    'stopped'       : 'Discontinued',
    'worsening'     : 'Worsening',
    'treatment resistant': 'Treatment Resistant',
    'no improvement': 'No Improvement',
    'seeking'       : 'Seeking Medical Advice',
    'ongoing'       : 'Ongoing',
}

def normalise_outcome(raw):
    if pd.isna(raw) or str(raw).strip().lower() in ('', 'nan', 'none', 'not specified'):
        return None
    t = str(raw).lower()
    # Pick the first matching category (ordered by clinical severity)
    for key, label in OUTCOME_MAP.items():
        if key in t:
            return label
    # Keep short raw values (≤ 30 chars), trim long ones
    cleaned = str(raw).strip().title()
    return cleaned[:30] if len(cleaned) <= 30 else None

llm_long['outcome_norm'] = llm_long['outcome'].apply(normalise_outcome)

outcome_counts = (
    llm_long['outcome_norm'].dropna()
    .value_counts()
    .head(12)
)

# ── Normalise causality hints into short temporal/mechanistic tags ────────────
CAUSALITY_PATTERNS = {
    'Within days of starting'   : r'\b\d+\s*days?\s*after\s*start|\bwithin\s*\d+\s*days?\b',
    'Within weeks of starting'  : r'\b\d+\s*weeks?\s*after\s*start|\bfirst\s*\d+\s*weeks?\b',
    'After dose increase'       : r'increas\w*\s*dos|dose\s*increas|uptitrat',
    'After dose decrease/taper' : r'decreas\w*\s*dos|taper|reduc\w*\s*dos',
    'Dechallenge (stopped→improved)': r'stop\w*.*improv|discon\w*.*improv|ceas\w*.*improv|quit\w*.*improv',
    'Rechallenge (restarted→returned)': r'restar\w*|reinstat\w*|re-start',
    'Temporal: months after'    : r'\b\d+\s*months?\s*after|\bafter\s*\d+\s*months?\b',
    'Temporal: same day/hours'  : r'\bsame\s*day\b|\bhours?\s*(after|later)\b',
    'Withdrawal on stopping'    : r'withdraw|brain\s*zap|discontinu\w*\s*syndrome',
    'Serotonin syndrome'        : r'serotonin\s*syndrome',
    'COVID / illness trigger'   : r'covid|after\s*illness|after\s*infection|after\s*surgery',
}

def tag_causality(raw):
    if pd.isna(raw) or str(raw).strip().lower() in ('', 'nan', 'none'):
        return []
    t = str(raw).lower()
    return [label for label, pattern in CAUSALITY_PATTERNS.items()
            if pd.Series([t]).str.contains(pattern, regex=True, na=False).iloc[0]]

causality_tags = llm_long['causality_hints'].apply(tag_causality).explode().dropna()
causality_counts = causality_tags[causality_tags != ''].value_counts()

# ── Figure ────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('LLM Contextual Richness: Outcome & Causality Profiling',
             fontweight='bold', fontsize=13)

# Panel A — Outcomes
colors_a = plt.cm.RdYlGn(np.linspace(0.15, 0.85, len(outcome_counts)))[::-1]
bars_a = ax1.barh(outcome_counts.index[::-1], outcome_counts.values[::-1],
                  color=colors_a[::-1], edgecolor='white', height=0.65)
for bar in bars_a:
    ax1.text(bar.get_width() + outcome_counts.values.max() * 0.01,
             bar.get_y() + bar.get_height() / 2,
             f'{int(bar.get_width()):,}',
             va='center', fontsize=9, color='#333333')
ax1.set_xlabel('Number of posts', fontsize=10)
ax1.set_title('Panel A — Patient-Reported Outcomes\n(LLM-extracted, normalised)', fontsize=10)
ax1.set_xlim(0, outcome_counts.values.max() * 1.18)
ax1.tick_params(axis='y', labelsize=10)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Panel B — Causality tags
colors_b = plt.cm.Blues(np.linspace(0.35, 0.85, len(causality_counts)))[::-1]
bars_b = ax2.barh(causality_counts.index[::-1], causality_counts.values[::-1],
                  color=colors_b[::-1], edgecolor='white', height=0.65)
for bar in bars_b:
    ax2.text(bar.get_width() + causality_counts.values.max() * 0.01,
             bar.get_y() + bar.get_height() / 2,
             f'{int(bar.get_width()):,}',
             va='center', fontsize=9, color='#333333')
ax2.set_xlabel('Number of posts matching pattern', fontsize=10)
ax2.set_title('Panel B — Causality Hint Patterns\n(temporal & mechanistic tags)', fontsize=10)
ax2.set_xlim(0, causality_counts.values.max() * 1.18)
ax2.tick_params(axis='y', labelsize=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig_outcome_causality.png'), dpi=300, bbox_inches='tight')
plt.show()

print('Outcome category counts:')
print(outcome_counts.to_string())
print('\nCausality tag counts:')
print(causality_counts.to_string())


In [ ]:
# ── 7.2 Off-Label Use and Drug-Drug Interactions ──────────────────────────────
llm_long['off_label_flag'] = llm_long['off_label_use'].str.lower().str.contains(
    'true|yes|off.label', na=False, regex=True
)
llm_long['ddi_flag'] = llm_long['drug_drug_interaction'].str.lower().str.contains(
    'interact|combination|contraindic', na=False, regex=True
)

off_label_by_cat = (
    llm_long.groupby('category')['off_label_flag']
    .agg(['mean', 'sum', 'count'])
    .rename(columns={'mean': 'proportion', 'sum': 'n_off_label', 'count': 'total'})
    .sort_values('proportion', ascending=False)
)

print('=== OFF-LABEL USE BY DRUG CATEGORY ===')
print(off_label_by_cat.to_string())

ddi_total = llm_long['ddi_flag'].mean()
print(f'\nProportion of posts flagging drug-drug interactions: {ddi_total:.1%}')

In [ ]:
# ── 7.3 Organ System Coverage by Method (Grouped Bar) ─────────────────────────
orgmap_llm  = llm_long[llm_long['organ_system_llm'] != 'Novel/Unlabelled']['organ_system_llm'].value_counts(normalize=True)
orgmap_dict = dict_long['organ_system_dict'].value_counts(normalize=True)

organs_all = sorted(set(orgmap_llm.index) | set(orgmap_dict.index))
pct_llm    = [orgmap_llm.get(o, 0) * 100 for o in organs_all]
pct_dict   = [orgmap_dict.get(o, 0) * 100 for o in organs_all]

x = np.arange(len(organs_all))
w = 0.4
fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x - w/2, pct_llm,  w, label='LLM',        color='steelblue',  alpha=0.85)
ax.bar(x + w/2, pct_dict, w, label='Dictionary',   color='darkorange', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(organs_all, rotation=40, ha='right')
ax.set_ylabel('% of All AEs')
ax.set_title('Organ System Coverage: LLM vs Dictionary', fontweight='bold')
ax.legend()
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig_organ_system_coverage.png'), dpi=300, bbox_inches='tight')
plt.show()

---
## Section 8: Longitudinal Patient-Level Analysis

In [ ]:
# ── 8.1 Filter Authors with >2 Posts and Compute Trajectories ─────────────────
llm_long['created_date'] = pd.to_datetime(llm_long['created_date'], errors='coerce')

author_post_counts = llm_long.groupby('author')['reddit_post_id'].nunique()
repeat_authors     = author_post_counts[author_post_counts > 2].index
longitudinal       = llm_long[llm_long['author'].isin(repeat_authors)].copy()
longitudinal       = longitudinal.sort_values(['author', 'created_date'])
longitudinal.to_csv(os.path.join(OUT, 'drug_ae_longitudinal_authors_gt2.csv'), index=False)

print('=== LONGITUDINAL COHORT ===')
print(f'Total authors (≥1 post):    {len(author_post_counts):,}')
print(f'Repeat authors (>2 posts):  {len(repeat_authors):,}')
print(f'LLM records (repeat):       {len(longitudinal):,}')
print(f'Unique drugs (repeat):      {longitudinal["drug"].nunique()}')
print(f'Unique AEs (repeat):        {longitudinal["ae_llm"].nunique():,}')

In [ ]:
# ── 8.2 Time-to-AE-Report Analysis by Drug Category ──────────────────────────
# For each (author, drug) pair:
#   first_drug_date = earliest post in the FULL LLM corpus mentioning that drug
#   first_ae_date   = earliest post where the same author reports an AE for that drug
# Delta (months) = latency between starting to discuss a drug and first reporting harm.

first_drug_date = (
    llm_long.groupby(['author', 'drug'])['created_date']
    .min().reset_index(name='first_drug_date')
)
first_ae_date = (
    longitudinal.groupby(['author', 'drug'])['created_date']
    .min().reset_index(name='first_ae_date')
)

ttr = first_drug_date.merge(first_ae_date, on=['author', 'drug'])
ttr['time_to_report_days']   = (ttr['first_ae_date'] - ttr['first_drug_date']).dt.days
ttr['time_to_report_months'] = ttr['time_to_report_days'] / 30.44
ttr['category'] = ttr['drug'].map(generic2cat).fillna('Unknown')
ttr = ttr[(ttr['category'] != 'Unknown') & (ttr['time_to_report_days'] >= 0)]

# Summary stats per category
summary = (
    ttr.groupby('category')['time_to_report_months']
    .agg(n='count', median='median',
         q1=lambda x: np.percentile(x, 25),
         q3=lambda x: np.percentile(x, 75))
    .reset_index()
    .sort_values('median', ascending=False)   # top = slowest → bottom = fastest
)
print(f'Author–drug pairs with valid time-to-AE data: {len(ttr):,}\n')
print(summary.round(1).to_string(index=False))

# Kruskal-Wallis test
groups = [g['time_to_report_months'].dropna().values
          for _, g in ttr.groupby('category') if len(g) >= 5]
p_kw = None
if len(groups) >= 2:
    stat_kw, p_kw = kruskal(*groups)
    print(f'\nKruskal-Wallis H = {stat_kw:.2f}, p = {p_kw:.4g}')

# ── Time bucket classification ────────────────────────────────────────────────
def bucket(m):
    if m == 0:              return 'Same post'
    elif m <= 1:            return '< 1 month'
    elif m <= 3:            return '1 – 3 months'
    elif m <= 6:            return '3 – 6 months'
    elif m <= 12:           return '6 – 12 months'
    else:                   return '> 12 months'

BUCKET_ORDER  = ['Same post', '< 1 month', '1 – 3 months',
                 '3 – 6 months', '6 – 12 months', '> 12 months']
BUCKET_COLORS = ['#1a9641', '#a6d96a', '#ffffbf', '#fdae61', '#f46d43', '#d73027']

ttr['bucket'] = ttr['time_to_report_months'].apply(bucket)

# Percentage breakdown per category
bucket_pct = (
    ttr.groupby(['category', 'bucket'])
    .size().reset_index(name='count')
)
bucket_pct['total'] = bucket_pct.groupby('category')['count'].transform('sum')
bucket_pct['pct']   = 100 * bucket_pct['count'] / bucket_pct['total']
bucket_pivot = (
    bucket_pct.pivot(index='category', columns='bucket', values='pct')
    .reindex(columns=BUCKET_ORDER)
    .fillna(0)
    .reindex(summary['category'])       # keep sorted by median
)

# ── Figure: stacked horizontal bars (top) + CDF curves (bottom) ──────────────
fig = plt.figure(figsize=(13, 12))
gs  = fig.add_gridspec(2, 1, hspace=0.45)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])

fig.suptitle(
    'Time from First Drug Mention to First AE Report on Reddit\n'
    'by Weight-loss Category (Repeat Authors Only)',
    fontweight='bold', fontsize=13, y=0.98)

# ── Panel A: 100 % stacked horizontal bar ────────────────────────────────────
lefts = np.zeros(len(bucket_pivot))
cat_labels = bucket_pivot.index.tolist()
n_map = summary.set_index('category')['n'].to_dict()
med_map = summary.set_index('category')['median'].to_dict()

for col, color in zip(BUCKET_ORDER, BUCKET_COLORS):
    vals = bucket_pivot[col].values
    bars = ax1.barh(cat_labels, vals, left=lefts, color=color,
                    label=col, height=0.62, edgecolor='white', linewidth=0.5)
    # Label segments ≥ 8 %
    for j, (v, l) in enumerate(zip(vals, lefts)):
        if v >= 8:
            ax1.text(l + v / 2, j, f'{v:.0f}%',
                     ha='center', va='center', fontsize=8.5,
                     color='black', fontweight='bold')
    lefts += vals

# Category labels with n= and median annotation on right side
for j, cat in enumerate(cat_labels):
    ax1.text(102, j,
             f"n={n_map[cat]:,}   med={med_map[cat]:.1f} mo",
             va='center', fontsize=9, color='#333333')

ax1.set_xlim(0, 100)
ax1.set_xlabel('Percentage of authors (%)', fontsize=11)
ax1.set_title('Panel A — How long until the author first reports an AE?\n'
              '(Each bar = 100 % of authors in that drug category, split by time bucket)',
              fontsize=10, pad=8)
ax1.legend(title='Time to first AE report', loc='lower right',
           bbox_to_anchor=(1.42, 0), fontsize=9, framealpha=0.9)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.spines['left'].set_visible(False)
ax1.tick_params(axis='y', length=0, labelsize=11)

# ── Panel B: Empirical CDF per category ──────────────────────────────────────
x_max = min(ttr['time_to_report_months'].quantile(0.95), 36)
palette = dict(zip(summary['category'],
                   plt.cm.Set2(np.linspace(0, 1, len(summary)))))

for cat in summary['category']:
    vals = ttr[ttr['category'] == cat]['time_to_report_months'].dropna()
    xs = np.sort(vals[vals <= x_max])
    ys = np.arange(1, len(xs) + 1) / len(vals) * 100
    ax2.plot(xs, ys, label=f"{cat}  (n={len(vals):,}, med={med_map[cat]:.1f} mo)",
             color=palette[cat], lw=2.5, alpha=0.9)

for thresh, lbl in [(1, '1 mo'), (3, '3 mo'), (6, '6 mo'), (12, '1 yr')]:
    ax2.axvline(thresh, color='#888888', linestyle=':', lw=1.2, alpha=0.6)
    ax2.text(thresh + 0.15, 97, lbl, fontsize=8, color='#666666', va='top')

ax2.set_xlim(0, x_max)
ax2.set_ylim(0, 103)
ax2.set_xlabel('Months from first drug mention to first AE report', fontsize=11)
ax2.set_ylabel('Cumulative % of authors\nwho have reported an AE', fontsize=11)
ax2.set_title('Panel B — Cumulative speed of AE reporting per category\n'
              '(steeper = authors report AEs sooner after mentioning the drug)',
              fontsize=10, pad=8)
ax2.legend(fontsize=9, loc='lower right', framealpha=0.9)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

if p_kw is not None:
    ax2.text(0.01, 0.97,
             f'Kruskal–Wallis: H = {stat_kw:.1f}, p = {p_kw:.3g}',
             transform=ax2.transAxes, va='top', fontsize=9,
             bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='grey', alpha=0.85))

plt.savefig(os.path.join(OUT, 'fig_time_to_ae_report.png'), dpi=300, bbox_inches='tight')
plt.show()

ttr[['author', 'drug', 'category', 'time_to_report_months', 'bucket']].to_csv(
    os.path.join(OUT, 'drug_ae_longitudinal_authors_gt2.csv'), index=False)
print('Saved: fig_time_to_ae_report.png  |  drug_ae_longitudinal_authors_gt2.csv')


In [ ]:
# ── 8.3 AE Persistence — Authors Reporting Same AE Across ≥2 Posts ────────────
ae_persistence = (
    longitudinal
    .groupby(['author', 'drug', 'ae_llm'])['reddit_post_id']
    .nunique().reset_index(name='n_posts_reporting_ae')
)
persistent = ae_persistence[ae_persistence['n_posts_reporting_ae'] >= 2]

print(f'Author-drug-AE combinations reported in ≥2 separate posts: {len(persistent):,}')
print(f'\nTop 15 most persistently reported AEs:')
print(
    persistent.groupby('ae_llm')['n_posts_reporting_ae']
    .sum().sort_values(ascending=False).head(15).to_string()
)

---
## Section 9: Statistical Validation and Robustness

In [ ]:
# ── 9.1 Multiple Testing FDR Summary Across All Tests ─────────────────────────
# Collect all raw p-values produced in this analysis for FDR correction

# Safe scalar extractor: handles numpy scalars, 0-d arrays, and plain floats
_pf = lambda x: float(np.asarray(x).ravel()[0])

all_tests = []

# From Section 4 (post-level comparison) — add if variable exists
try:
    all_tests.append({'test': 'Wilcoxon: AE/post LLM vs Dict', 'p_raw': _pf(p)})
except NameError:
    pass

# From Section 6 (lead time)
try:
    all_tests.append({'test': 'Wilcoxon: lead time ≠ 0', 'p_raw': _pf(p_w)})
except NameError:
    pass

# From Section 8 (time to report)
try:
    all_tests.append({'test': 'Kruskal-Wallis: TTR by category', 'p_raw': _pf(p_kw)})
except NameError:
    pass

if len(all_tests) >= 2:
    tests_df = pd.DataFrame(all_tests)
    _, q_vals, _, _ = multipletests(tests_df['p_raw'].astype(float).to_numpy(), method='fdr_bh')
    tests_df['q_fdr'] = q_vals
    tests_df['significant_fdr'] = tests_df['q_fdr'] < 0.05
    tests_df.to_csv(os.path.join(OUT, 'statistical_test_summary.csv'), index=False)
    print('=== GLOBAL FDR-CORRECTED TEST SUMMARY ===')
    print(tests_df.to_string(index=False))
else:
    print('Fewer than 2 tests collected — FDR correction not required.')


In [ ]:
# ── 9.2 Sensitivity Analysis — Underreported AE Thresholds ────────────────────
sens_results = []
for min_p in [5, 10, 20]:
    for min_a in [2, 3, 5]:
        mask = ((underrep['reddit_post_count'] >= min_p) &
                (underrep['unique_authors']    >= min_a) &
                (underrep['faers_count'] == 0))
        sens_results.append({
            'min_posts': min_p, 'min_authors': min_a,
            'n_faers_absent': mask.sum()
        })
sens_df = pd.DataFrame(sens_results)
sens_df.to_csv(os.path.join(OUT, 'sensitivity_underreported_aes.csv'), index=False)
print('=== SENSITIVITY ANALYSIS: THRESHOLD IMPACT ON FAERS-ABSENT AES ===')
print(sens_df.pivot(index='min_posts', columns='min_authors', values='n_faers_absent').to_string())

In [ ]:
# ── 9.3 Inter-Method Agreement — Cohen's Kappa ────────────────────────────────
# For posts processed by both methods, measure binary agreement (AE detected: yes/no)

posts_all = set(reddit['reddit_post_id'])
posts_llm_ae  = set(llm_long['reddit_post_id'])
posts_dict_ae = set(dict_long['reddit_post_id'])

# Only posts where at least one method detected something
posts_either = posts_llm_ae | posts_dict_ae
sample_posts = list(posts_either)[:5000]  # Cap for speed

y_llm  = [1 if p in posts_llm_ae  else 0 for p in sample_posts]
y_dict = [1 if p in posts_dict_ae else 0 for p in sample_posts]

kappa = cohen_kappa_score(y_llm, y_dict)
print(f'Cohen\'s Kappa (AE detected: LLM vs Dictionary): κ = {kappa:.3f}')

kappa_interp = ('Slight' if kappa < 0.2 else
                'Fair' if kappa < 0.4 else
                'Moderate' if kappa < 0.6 else
                'Substantial' if kappa < 0.8 else
                'Almost Perfect')
print(f'Interpretation: {kappa_interp}')

---
## Section 10: Publication-Ready Summary Tables and Figures

In [ ]:
# ── Table 1: Dataset Characteristics ──────────────────────────────────────────
table1 = pd.DataFrame([
    ['Reddit posts (raw)',                     f'{len(reddit_raw):,}'],
    ['Reddit posts (after QC)',                f'{len(reddit):,}'],
    ['  Date range',                           f"{reddit['created_date'].min().date()} – {reddit['created_date'].max().date()}"],
    ['  Unique authors',                       f"{reddit['author'].nunique():,}"],
    ['  Unique subreddits',                    f"{reddit['subreddit'].nunique():,}"],
    ['FAERS records',                          f'{len(faers_norm):,}'],
    ['  FAERS year range',                     f"{faers_norm['year'].min()} – {faers_norm['year'].max()}"],
    ['  FAERS unique drugs',                   f"{faers_norm['drug'].nunique():,}"],
    ['  FAERS unique AEs',                     f"{faers_norm['ae'].nunique():,}"],
    ['Reference weight-losss (generics)',   f'{len(all_generics)}'],
    ['AE dictionary terms',                    f'{len(ae_dict_terms):,}'],
], columns=['Characteristic', 'Value'])

table1.to_csv(os.path.join(OUT, 'Table1_dataset_characteristics.csv'), index=False)
display(table1)

In [ ]:
# ── Table 2: AE Extraction Coverage Comparison ────────────────────────────────
post_llm_med  = llm_long.groupby('reddit_post_id')['ae_llm'].count().median()
post_dict_med = dict_long.groupby('reddit_post_id')['ae_dict'].count().median()

table2 = pd.DataFrame([
    ['Unique AE terms',                      f'{len(set_ae_llm):,}',   f'{len(set_ae_dict):,}'],
    ['Unique drug-AE pairs',                 f'{len(set_pair_llm):,}', f'{len(set_pair_dict):,}'],
    ['Posts with ≥1 AE (%)',
     f'{100*n_has_both/len(reddit):.1f}%',  f'{100*n_dict_both/len(reddit):.1f}%'],
    ['Median AEs per post (IQR)',
     f'{post_llm_med:.1f}',                  f'{post_dict_med:.1f}'],
    ['Novel AEs (not in dictionary or FAERS)',f'{len(novel_llm_aes):,}', 'N/A (by definition)'],
    ['AEs mapped to organ system',
     f'{llm_long["dict_matched"].sum():,} ({100*llm_long["dict_matched"].mean():.1f}%)',
     f'{len(dict_long):,} (100%)'],
], columns=['Metric', 'LLM (Method A)', 'Dictionary (Method B)'])

table2.to_csv(os.path.join(OUT, 'Table2_coverage_comparison.csv'), index=False)
display(table2)

In [ ]:
# ── Table 3: Signal Lead Time Summary ─────────────────────────────────────────
table3 = pd.DataFrame([
    ['Matched drug-AE pairs (Reddit ∩ FAERS)',
     f'{len(lead_df):,}'],
    ['% where Reddit (LLM) leads FAERS',
     f'{100*leads_reddit_first/len(lead_df):.1f}%'],
    ['Median lead time (years)',
     f'{lead_df["lead_time_years"].median():.1f}'],
    ['Mean lead time (years)',
     f'{lead_df["lead_time_years"].mean():.2f}'],
    ['IQR lead time',
     f'[{lead_df["lead_time_years"].quantile(0.25):.1f}, {lead_df["lead_time_years"].quantile(0.75):.1f}]'],
    ['Max lead time (years)',
     f'{lead_df["lead_time_years"].max():.0f}'],
    ['Wilcoxon signed-rank p-value',
     f'{p_w:.4g}' if 'p_w' in dir() else 'N/A'],
    ['Bootstrap 95% CI for median',
     f'[{ci_lo:.2f}, {ci_hi:.2f}]' if 'ci_lo' in dir() else 'N/A'],
], columns=['Statistic', 'Value'])

table3.to_csv(os.path.join(OUT, 'Table3_lead_time_summary.csv'), index=False)
display(table3)

In [ ]:
# ── Table 4: Top 20 Underreported AEs by LLM ─────────────────────────────────
# First Reddit date per pair for the table
first_dates = (
    llm_long.groupby(['drug', 'ae_llm'])['created_date']
    .min().reset_index()
    .rename(columns={'ae_llm': 'ae', 'created_date': 'first_reddit_date'})
)
first_dates['first_reddit_date'] = first_dates['first_reddit_date'].dt.date

table4 = (
    underrep_filtered[underrep_filtered['faers_status'] != 'Reported']
    .merge(first_dates, on=['drug', 'ae'], how='left')
    .sort_values('reddit_post_count', ascending=False)
    .head(20)
    [['drug', 'ae', 'reddit_post_count', 'unique_authors',
      'faers_count', 'first_reddit_date', 'first_year_faers', 'faers_status']]
    .rename(columns={
        'drug': 'Drug', 'ae': 'Adverse Event',
        'reddit_post_count': 'Reddit Posts', 'unique_authors': 'Unique Authors',
        'faers_count': 'FAERS Count', 'first_reddit_date': 'First Reddit Date',
        'first_year_faers': 'First FAERS Year', 'faers_status': 'FAERS Status'
    })
)

table4.to_csv(os.path.join(OUT, 'Table4_top20_underreported_aes.csv'), index=False)
display(table4)

In [ ]:
# ── Figure 5: Reddit Monthly Timeline by Drug Category with COVID Annotation ──
import matplotlib.dates as mdates

reddit_cat2 = reddit.copy()
reddit_cat2['year_month'] = reddit_cat2['created_date'].dt.to_period('M')

# Use LLM-assigned category at post level (take first category per post)
post_cat_map = (
    llm_long[['reddit_post_id', 'category']]
    .drop_duplicates('reddit_post_id')
    .set_index('reddit_post_id')['category']
)
reddit_cat2['category'] = reddit_cat2['reddit_post_id'].map(post_cat_map)
reddit_cat2 = reddit_cat2.dropna(subset=['category'])
reddit_cat2 = reddit_cat2[reddit_cat2['category'] != 'Unknown']

monthly_cat = (
    reddit_cat2.groupby(['year_month', 'category'])['reddit_post_id']
    .nunique().reset_index(name='post_count')
)
pivot = monthly_cat.pivot(index='year_month', columns='category', values='post_count').fillna(0)

# Convert PeriodIndex → timestamps so matplotlib uses proper date scale
pivot.index = pivot.index.to_timestamp()

fig, ax = plt.subplots(figsize=(16, 7))
pivot.plot(ax=ax, linewidth=2)

events = [
    ('COVID-19 Declared', '2020-03'),
    ('Vaccine Rollout',   '2021-01'),
    ('WHO Pandemic End',  '2023-05'),
]
ymax = pivot.values.max()
for name, date in events:
    dt = pd.Timestamp(date)
    ax.axvline(dt, color='grey', linestyle='--', lw=1.5, alpha=0.7)
    ax.text(dt, ymax * 0.92, name, rotation=90, va='top', ha='right',
            fontsize=9, color='grey')

# Yearly ticks — avoid the default 120-month label clutter
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

ax.set_xlabel('Year')
ax.set_ylabel('Unique Posts per Month')
ax.set_title('Figure 5: Monthly Reddit Post Count by Weight-loss Category (2021–2025)',
             fontweight='bold')
ax.legend(title='Drug Category', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig5_reddit_timeline_by_category.png'), dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# ── Figure 6: Top PRR Signals — Horizontal Bar Chart ────────────────────────
from matplotlib.patches import Patch

if len(prr_df) > 0 and 'signal_fdr' in prr_df.columns:
    sig = prr_df[prr_df['signal_fdr']].copy()
    n_inf = (sig['prr'] == np.inf).sum()

    # Rank by post-count evidence (most robust signals first); break ties by PRR
    sig_finite = sig[np.isfinite(sig['prr'])].copy()
    sig_inf    = sig[~np.isfinite(sig['prr'])].copy()

    # Take top 30: prefer finite PRR signals; if <30 available also pull from inf pool
    top_finite = sig_finite.nlargest(30, 'n_ae_drug')
    if len(top_finite) < 30:
        extra = sig_inf.nlargest(30 - len(top_finite), 'n_ae_drug')
        extra['prr'] = 999.0   # display cap for inf
        top_prr = pd.concat([top_finite, extra], ignore_index=True)
    else:
        top_prr = top_finite

    if len(top_prr) > 0:
        top_prr = top_prr.sort_values('prr', ascending=True)
        top_prr['label'] = top_prr['drug'].str.capitalize() + '  ➜  ' + top_prr['ae']

        # Colour by drug name (category not in prr_df)
        drug_names = top_prr['drug'].unique()
        cmap_colors = plt.cm.tab20.colors
        palette = {d: cmap_colors[i % len(cmap_colors)] for i, d in enumerate(drug_names)}
        colors = [palette[d] for d in top_prr['drug']]

        fig, ax = plt.subplots(figsize=(13, max(8, len(top_prr) * 0.45)))
        bars = ax.barh(top_prr['label'], top_prr['prr'], color=colors, edgecolor='white', height=0.7)

        # Annotate each bar with n and q-value; mark capped inf bars
        for bar, (_, row) in zip(bars, top_prr.iterrows()):
            cap_note = ' [excl.]' if row['prr'] == 999.0 else ''
            ax.text(bar.get_width() + top_prr['prr'].max() * 0.01,
                    bar.get_y() + bar.get_height() / 2,
                    f"n={int(row['n_ae_drug'])}  q={row['q_value']:.2e}{cap_note}",
                    va='center', ha='left', fontsize=8, color='#333333')

        # Signal threshold reference line
        ax.axvline(2.0, color='red', linestyle='--', lw=1.2, alpha=0.7)
        ax.text(2.05, ax.get_ylim()[1] * 0.995, 'PRR = 2',
                color='red', fontsize=8, va='top')

        ax.set_xlabel('Proportional Reporting Ratio (PRR)', fontsize=11)
        ax.set_ylabel('')
        title_suffix = f'\n({n_inf:,} exclusive signals [excl.] capped at 999)' if n_inf > 0 else ''
        ax.set_title(
            f'Figure 6: Top {len(top_prr)} PRR Signals — LLM ADE Extraction\n'
            f'(FDR q < 0.05, PRR ≥ 2, n ≥ 3){title_suffix}',
            fontweight='bold', fontsize=11)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.set_xlim(0, top_prr['prr'].max() * 1.20)

        # Drug colour legend
        legend_handles = [Patch(facecolor=palette[d], label=d.capitalize()) for d in drug_names]
        ax.legend(handles=legend_handles, title='Drug', bbox_to_anchor=(1.01, 1),
                  loc='upper left', fontsize=9)

        plt.tight_layout()
        plt.savefig(os.path.join(OUT, 'fig6_prr_top_signals.png'), dpi=300, bbox_inches='tight')
        plt.show()
        print(f'Figure 6 saved — {len(top_prr)} signals plotted '
              f'({n_inf:,} exclusive inf-PRR signals in full dataset).')
    else:
        print('No FDR-significant PRR signals to plot.')
else:
    print('PRR analysis skipped or no signals detected.')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Section 11: Stigmatised & Sensitive Adverse Events — LLM vs FAERS
# ══════════════════════════════════════════════════════════════════════════════
# Hypothesis: LLM extraction captures clinically and socially sensitive ADEs
# (e.g. sexual dysfunction, suicidality, self-harm, addiction, psychosis)
# that patients disclose anonymously on Reddit but systematically under-report
# to healthcare providers and therefore remain invisible in FAERS.
# ══════════════════════════════════════════════════════════════════════════════

# ── 11.1  Define sensitive ADE categories with keyword patterns ───────────────
SENSITIVE_CATEGORIES = {
    'Sexual Dysfunction / PSSD': [
        'sexual', 'libido', 'orgasm', 'ejacul', 'erect', 'pssd',
        'anorgasm', 'genital', 'arousal', 'frigid', 'impoten',
        'decreased sex drive', 'loss of libido', 'sexual dysfunction',
    ],
    'Suicidality / Self-Harm': [
        'suicid', 'self-harm', 'self harm', 'self-injur', 'cutting',
        'overdos', 'ideation', 'passive death wish', 'want to die',
        'attempted suicide', 'suicidal thought',
    ],
    'Withdrawal / Discontinuation': [
        'withdraw', 'discontinu', 'brain zap', 'brain zaps', 'taper',
        'rebound', 'discontinuation syndrome', 'cessation',
    ],
    'Substance Misuse / Addiction': [
        'abuse', 'misuse', 'addict', 'dependenc', 'craving',
        'recreational', 'tolerance', 'drug abuse',
    ],
    'Psychosis / Dissociation': [
        'hallucin', 'paranoi', 'psychos', 'dissociat', 'depersonali',
        'dereali', 'psychotic', 'delusio', 'mania', 'manic episode',
    ],
    'Emotional Blunting / Apathy': [
        'emotional blunt', 'emotional numb', 'apathy', 'anhedonia',
        'flat affect', 'blunted', 'numbing', 'emotional flatness',
    ],
    'Aggression / Impulsivity': [
        'aggress', 'impuls', 'rage', 'violent', 'homicid',
        'anger outburst', 'irritabilit',
    ],
}

def classify_sensitive(ae_term, category_map=SENSITIVE_CATEGORIES):
    """Return the first matching sensitive category label, or None."""
    t = str(ae_term).lower()
    for cat, keywords in category_map.items():
        if any(k in t for k in keywords):
            return cat
    return None

# Apply classification to the full LLM longitudinal table
llm_long['sensitive_cat'] = llm_long['ae_llm'].apply(classify_sensitive)
sensitive_llm = llm_long[llm_long['sensitive_cat'].notna()].copy()

print(f'Total LLM ADE observations        : {len(llm_long):,}')
print(f'Sensitive ADE observations (LLM)  : {len(sensitive_llm):,} '
      f'({100*len(sensitive_llm)/len(llm_long):.1f}%)')
print(f'Unique sensitive AE terms (LLM)   : {sensitive_llm["ae_llm"].nunique():,}')
print(f'Unique authors reporting sensitive : {sensitive_llm["author"].nunique():,}')
print()
print('--- Breakdown by sensitive category ---')
cat_counts = (
    sensitive_llm.groupby('sensitive_cat')
    .agg(n_obs=('ae_llm', 'count'),
         n_unique_aes=('ae_llm', 'nunique'),
         n_authors=('author', 'nunique'),
         n_drugs=('drug', 'nunique'))
    .sort_values('n_obs', ascending=False)
)
print(cat_counts.to_string())

# ── 11.2  Classify FAERS sensitive terms using the same taxonomy ──────────────
faers_norm2 = faers_norm.copy() if 'faers_norm' in dir() else faers.copy()
faers_norm2.columns = faers_norm2.columns.str.lower().str.strip()
faers_ae_col  = 'ae'   if 'ae'   in faers_norm2.columns else faers_norm2.columns[1]
faers_drg_col = 'drug' if 'drug' in faers_norm2.columns else faers_norm2.columns[0]

faers_norm2['sensitive_cat'] = faers_norm2[faers_ae_col].apply(classify_sensitive)
sensitive_faers = faers_norm2[faers_norm2['sensitive_cat'].notna()].copy()

faers_cat_counts = (
    sensitive_faers.groupby('sensitive_cat')
    .agg(n_faers_records=('sensitive_cat', 'count'),
         n_unique_aes=(faers_ae_col, 'nunique'))
    .sort_values('n_faers_records', ascending=False)
)
print('\n--- FAERS sensitive category breakdown ---')
print(faers_cat_counts.to_string())

# ── 11.3  Per-category comparison table ───────────────────────────────────────
compare_rows = []
for cat in SENSITIVE_CATEGORIES:
    llm_sub   = sensitive_llm[sensitive_llm['sensitive_cat'] == cat]
    faers_sub = sensitive_faers[sensitive_faers['sensitive_cat'] == cat]

    llm_pairs   = set(zip(llm_sub['drug'].str.lower(), llm_sub['ae_llm'].str.lower()))
    faers_pairs = set(zip(faers_sub[faers_drg_col].str.lower(), faers_sub[faers_ae_col].str.lower()))
    never_n   = len(llm_pairs - faers_pairs)
    pct_never = round(100 * never_n / len(llm_pairs), 1) if llm_pairs else 0.0

    compare_rows.append({
        'Sensitive Category'    : cat,
        'LLM Posts'             : len(llm_sub),
        'LLM Unique AE Terms'   : llm_sub['ae_llm'].nunique(),
        'LLM Unique Authors'    : llm_sub['author'].nunique(),
        'FAERS Records'         : len(faers_sub),
        'FAERS Unique AE Terms' : faers_sub[faers_ae_col].nunique(),
        'LLM Drug-AE Pairs'     : len(llm_pairs),
        'Never in FAERS (pairs)': never_n,
        '% Never in FAERS'      : pct_never,
    })

compare_df = pd.DataFrame(compare_rows).sort_values('LLM Posts', ascending=False)
compare_df.to_csv(os.path.join(OUT, 'sensitive_ae_llm_vs_faers.csv'), index=False)
print('\n=== SENSITIVE ADE COMPARISON: LLM vs FAERS ===')
print(compare_df.to_string(index=False))

# ── 11.4  Figure 7a: Grouped bar — LLM posts vs FAERS records (log scale) ─────
fig, ax = plt.subplots(figsize=(13, 6))
x, w = np.arange(len(compare_df)), 0.38
ax.bar(x - w/2, compare_df['LLM Posts'],    w, label='LLM (Reddit posts)',  color='#2196F3', alpha=0.88)
ax.bar(x + w/2, compare_df['FAERS Records'], w, label='FAERS records',       color='#FF5722', alpha=0.88)
ax.set_xticks(x)
ax.set_xticklabels(compare_df['Sensitive Category'], rotation=28, ha='right', fontsize=10)
ax.set_ylabel('Count (log scale)', fontsize=11)
ax.set_title('Figure 7a: Sensitive ADE Reporting — LLM (Reddit) vs FAERS\nby Stigmatised Category',
             fontweight='bold', fontsize=12)
ax.legend(fontsize=10)
ax.set_yscale('log')
ax.set_ylim(bottom=1)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{int(v):,}'))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig7a_sensitive_ae_llm_vs_faers.png'), dpi=300, bbox_inches='tight')
plt.show()

# ── 11.5  Figure 7b: % of drug–AE pairs entirely absent from FAERS ────────────
fig, ax = plt.subplots(figsize=(11, 5))
bar_colors = plt.cm.RdYlGn_r(np.linspace(0.15, 0.85, len(compare_df)))
bars = ax.barh(compare_df['Sensitive Category'], compare_df['% Never in FAERS'],
               color=bar_colors, edgecolor='white', height=0.65)
for bar, (_, row) in zip(bars, compare_df.iterrows()):
    ax.text(min(bar.get_width() + 0.5, 102),
            bar.get_y() + bar.get_height() / 2,
            f"{row['% Never in FAERS']:.1f}%  (n={row['Never in FAERS (pairs)']:,})",
            va='center', ha='left', fontsize=9)
ax.axvline(100, color='grey', linestyle=':', lw=1, alpha=0.6)
ax.set_xlim(0, 118)
ax.set_xlabel('% of LLM Drug–AE Pairs with Zero FAERS Reports', fontsize=11)
ax.set_title('Figure 7b: Proportion of Sensitive Drug–AE Pairs\nEntirely Absent from FAERS',
             fontweight='bold', fontsize=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig7b_sensitive_ae_never_in_faers.png'), dpi=300, bbox_inches='tight')
plt.show()

# ── 11.6  Top 20 sensitive ADEs never reported in FAERS ───────────────────────
faers_ae_set = set(faers_norm2[faers_ae_col].str.lower().dropna())
llm_counts_sensitive = (
    sensitive_llm.groupby(['sensitive_cat', 'ae_llm', 'drug'])['reddit_post_id']
    .nunique().reset_index(name='n_posts')
)
llm_counts_sensitive['in_faers'] = llm_counts_sensitive['ae_llm'].str.lower().isin(faers_ae_set)
never_faers_top = (
    llm_counts_sensitive[~llm_counts_sensitive['in_faers']]
    .sort_values('n_posts', ascending=False)
    .drop_duplicates(subset=['ae_llm', 'drug'])
    .head(20)
)
never_faers_top.to_csv(os.path.join(OUT, 'top_sensitive_aes_never_in_faers.csv'), index=False)
print('\n=== TOP 20 SENSITIVE ADEs NEVER REPORTED IN FAERS ===')
print(never_faers_top[['sensitive_cat', 'drug', 'ae_llm', 'n_posts']].to_string(index=False))

# ── 11.7  Figure 7c: Split into Drug-ADE pairs and Drug-Withdrawal pairs ──────
VERIFY_OUT = '/Volumes/Poornima/LLM_Antideprassant_26_03_2026/AE_Signal_Detection_Results_verify'
os.makedirs(VERIFY_OUT, exist_ok=True)

all_plot_data = never_faers_top.copy()
all_plot_data['label'] = all_plot_data['drug'].str.capitalize() + '  ➜  ' + all_plot_data['ae_llm']

cat_palette = {c: plt.cm.Set2.colors[i % 8] for i, c in enumerate(SENSITIVE_CATEGORIES)}

WITHDRAWAL_CAT = 'Withdrawal / Discontinuation'

# Split into two subsets
ade_data        = all_plot_data[all_plot_data['sensitive_cat'] != WITHDRAWAL_CAT].copy()
withdrawal_data = all_plot_data[all_plot_data['sensitive_cat'] == WITHDRAWAL_CAT].copy()

from matplotlib.patches import Patch as _Patch

def _plot_horiz_bar(data, title, filename):
    """Helper: horizontal bar chart for a subset of sensitive never-FAERS pairs."""
    if data.empty:
        print(f'  [skip] No data for: {title}')
        return
    data = data.sort_values('n_posts', ascending=True).copy()
    fig, ax = plt.subplots(figsize=(12, max(5, len(data) * 0.55 + 1.5)))
    ax.barh(data['label'],
            data['n_posts'],
            color=[cat_palette.get(c, '#AAAAAA') for c in data['sensitive_cat']],
            edgecolor='white', height=0.7)
    for i, (_, row) in enumerate(data.iterrows()):
        ax.text(row['n_posts'] + data['n_posts'].max() * 0.01, i,
                str(int(row['n_posts'])), va='center', fontsize=8)
    legend_handles = [_Patch(facecolor=cat_palette[c], label=c)
                      for c in SENSITIVE_CATEGORIES
                      if c in data['sensitive_cat'].values]
    ax.legend(handles=legend_handles, title='Category',
              bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
    ax.set_xlabel('Number of Reddit Posts', fontsize=11)
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlim(0, data['n_posts'].max() * 1.18)
    plt.tight_layout()
    plt.savefig(os.path.join(VERIFY_OUT, filename), dpi=300, bbox_inches='tight')
    plt.show()
    print(f'  ✓  {filename}  ({len(data)} pairs)')

# Figure 7c-ADE: Drug–ADE pairs (no withdrawal)
_plot_horiz_bar(
    ade_data,
    title='Figure 7c (ADE): Top Sensitive Drug–ADE Pairs Reported on Reddit\nbut Never Recorded in FAERS (Withdrawal Excluded)',
    filename='fig7c_ade_pairs_never_faers.png'
)

# Figure 7c-W: Drug–Withdrawal pairs only
_plot_horiz_bar(
    withdrawal_data,
    title='Figure 7c (Withdrawal): Drug–Withdrawal/Discontinuation Pairs\nReported on Reddit but Never Recorded in FAERS',
    filename='fig7c_withdrawal_pairs_never_faers.png'
)

print('\nSection 11 complete. Outputs saved:')
for f in ['sensitive_ae_llm_vs_faers.csv', 'top_sensitive_aes_never_in_faers.csv',
          'fig7a_sensitive_ae_llm_vs_faers.png',
          'fig7b_sensitive_ae_never_in_faers.png']:
    path = os.path.join(OUT, f)
    status = '✓' if os.path.exists(path) else '✗'
    print(f'  {status}  {f}')
for f in ['fig7c_ade_pairs_never_faers.png', 'fig7c_withdrawal_pairs_never_faers.png']:
    path = os.path.join(VERIFY_OUT, f)
    status = '✓' if os.path.exists(path) else '✗'
    print(f'  {status}  {f}  (→ AE_Signal_Detection_Results_verify/)')


In [ ]:
# ── Final Output Summary ──────────────────────────────────────────────────────
csv_files = [
    # Cleaned datasets
    'reddit_clean.csv',
    'faers_clean.csv',
    'faers_normalised.csv',
    # Drug-AE longitudinal tables
    'llm_drug_ae_longitudinal.csv',
    'dict_drug_ae_longitudinal.csv',
    'drug_ae_longitudinal_authors_gt2.csv',
    # Coverage & novelty
    'unique_pairs_comparison.csv',
    'novel_llm_only_aes.csv',
    'three_way_venn_counts.csv',
    # Signal detection
    'underreported_ae_llm_vs_faers.csv',
    'prr_llm_signals.csv',
    'signal_lead_time.csv',
    'its_results.csv',
    'sensitivity_underreported_aes.csv',
    'statistical_test_summary.csv',
    # Sensitive / stigmatised AEs
    'sensitive_ae_llm_vs_faers.csv',
    'top_sensitive_aes_never_in_faers.csv',
    # Summary tables
    'Table1_dataset_characteristics.csv',
    'Table2_coverage_comparison.csv',
    'Table3_lead_time_summary.csv',
    'Table4_top20_underreported_aes.csv',
]

fig_files = [
    'fig1_venn_llm_vs_dict.png',
    'fig2_lead_time_and_cumulative.png',
    'fig3_underreported_diverging.png',
    'fig4_kaplan_meier.png',
    'fig5_reddit_timeline_by_category.png',
    'fig6_prr_top_signals.png',
    'fig7a_sensitive_ae_llm_vs_faers.png',
    'fig7b_sensitive_ae_never_in_faers.png',
    'fig7c_top_sensitive_aes_never_faers.png',
    'fig_top_aes_comparison.png',
    'fig_organ_system_coverage.png',
    'fig_outcome_causality.png',
    'fig_time_to_ae_report.png',
]

all_files = csv_files + fig_files

print('=== STUDY OUTPUT FILES ===')
print(f'Output directory: {OUT}\n')

missing = 0
for f in all_files:
    path = os.path.join(OUT, f)
    exists = os.path.exists(path)
    if not exists:
        missing += 1
    size = f'{os.path.getsize(path)/1024:.0f} KB' if exists else 'MISSING'
    print(f'  {"✓" if exists else "✗"} {f:<55} {size}')

print(f'\nTotal: {len(all_files)} files  |  Present: {len(all_files) - missing}  |  Missing: {missing}')
print('\n=== ANALYSIS COMPLETE ===')
